# Simulador de Envio de Dados EEG via Serial 

Este notebook carrega dados de teste de EEG pré-processados, obtém os parâmetros de quantização de um modelo TFLite treinado, quantiza os dados e os envia via porta serial para um dispositivo ESP32, simulando a entrada de dados para inferência embarcada.

In [ ]:
import serial
import time
import numpy as np
import pickle
import random
# import threading 
# import msvcrt 
from inputimeout import inputimeout, TimeoutOccurred 

print("Bibliotecas importadas.")

Bibliotecas importadas.


In [ ]:
# config
PORTA_SERIAL = 'COM3' 
BAUDRATE = 115200
DADOS_PATH = 'Dados_EEG\dados_EEG1.pkl' 

input_scale = 3.9294116497039795
input_zero_point = 9
model_input_dtype = np.int8

In [4]:
#import pickle

with open(DADOS_PATH, mode='rb') as f:
  X_train, y_train, X_test, y_test, CANAIS, TAM_JANELA= pickle.load(f)

if X_test is not None:
  print(f"\nVerificando o tipo de dado de X_test: {X_test.dtype}")


if y_test is not None:
  cont_0 = np.sum(y_test == 0)
  cont_1 = np.sum(y_test == 1)
  cont_2 = np.sum(y_test == 2)
  
  print(f"\nClasse 0: {cont_0}, Classe 1: {cont_1}, Classe 2: {cont_2}")
else:
  print("erroo")


Verificando o tipo de dado de X_test: float64

Classe 0: 2070, Classe 1: 1029, Classe 2: 1041


In [5]:
def quantize_sample(sample_float_raw, scale, zero_point):

    sample_quantized = (sample_float_raw / scale) + zero_point
    sample_quantized = np.round(sample_quantized)
    sample_quantized = np.clip(sample_quantized, -128, 127) 
    
    return sample_quantized.astype(np.int8)

print("Função quantize_sample definida.")

Função quantize_sample definida.


In [ ]:
indices_por_classe = {}
if y_test is not None and X_test is not None:
    for i, classe in enumerate(y_test):
        
        classe_int = int(classe) 
        if classe_int not in indices_por_classe:
            indices_por_classe[classe_int] = []
        indices_por_classe[classe_int].append(i)

    print("\n--- Índice de Amostras por Classe ---")
    for classe, indices in indices_por_classe.items():
        print(f"Classe {classe}: {len(indices)} amostras disponíveis.")
else:
    print("\nAVISO: X_test ou y_test não foram carregados. O envio por classe falhará.")


--- Índice de Amostras por Classe ---
Classe 0: 2070 amostras disponíveis.
Classe 2: 1041 amostras disponíveis.
Classe 1: 1029 amostras disponíveis.


In [ ]:
def enviar_amostra_por_classe(ser, classe_desejada):
    """
    Prepara, envia uma amostra E LÊ A RESPOSTA do ESP32
    por uma porta serial JÁ ABERTA.
    Retorna True/False se o envio foi ok.
    """
    
    prefixo = "     " 
    
    if X_test is None or y_test is None:
        print(f"\n{prefixo}ERRO: Dados X_test/y_test não carregados.")
        return False
    if classe_desejada not in indices_por_classe or not indices_por_classe[classe_desejada]:
        print(f"\n{prefixo}AVISO: Nenhuma amostra encontrada para a Classe {classe_desejada}.")
        return False
    
    indices_disponiveis = indices_por_classe[classe_desejada]
    indice_aleatorio = random.choice(indices_disponiveis)
    print(f"\n{prefixo}--- Selecionado Amostra (Índice: {indice_aleatorio}) da Classe Real: {classe_desejada} ---")
    
    amostra_bruta = X_test[indice_aleatorio] 
    
    if model_input_dtype == np.int8:
        amostra_final_para_envio = quantize_sample(amostra_bruta.astype(np.float32), input_scale, input_zero_point)
        dtype_envio = "int8"
        
    else: 
        amostra_final_para_envio = amostra_bruta.astype(np.float32) 
        dtype_envio = "float32"
        
    dados_para_enviar_bytes = amostra_final_para_envio.flatten().tobytes()
    
    
    # HANDSHAKE 
    try:
        ser.flushInput()
        
        ser.write(b'?')
        ser.flush()

        resposta_ack = ser.read(1) 

        if resposta_ack == b'A':
            print(f"{prefixo}Permissão ('A') recebida. Enviando dados...")
            
            bytes_enviados = ser.write(dados_para_enviar_bytes)
            ser.flush()
            print(f"{prefixo}Bytes enviados: {bytes_enviados}.")
            
            print(f"{prefixo}Aguardando resultado da inferência (Timeout total: 10s)...")
            
            start_time = time.time()
            resposta_completa_str = ""
            resposta_recebida = False
            
            try:
                while (time.time() - start_time) < 10.0:
                    resposta_bytes = ser.readline() 
                    if resposta_bytes:
                        resposta_recebida = True
                        resposta_str = resposta_bytes.decode('utf-8').strip()
                        resposta_completa_str += resposta_str + "\n"
                        if "ESP32 pronto" in resposta_str:
                            break
                    else:
                        pass 
                
                if resposta_recebida:
                    print("\n--- [RESPOSTA DO ESP32] ---") 
                    print(resposta_completa_str, end='')
                    print("-----------------------------")
                    if "ESP32 pronto" not in resposta_completa_str:
                         print(f"\n{prefixo}AVISO: Resposta incompleta (não recebeu o 'ESP32 pronto' final).")
                else:
                    print(f"\n{prefixo}AVISO: Nenhuma resposta recebida do ESP32 (timeout total de 10s).")

            except UnicodeDecodeError:
                print(f"\n[RESPOSTA DO ESP32 (BINÁRIO)]\nRecebido lixo binário.\n")
            
        else:
            print(f"{prefixo}ERRO DE HANDSHAKE: ESP32 não respondeu 'A'. Respondeu: {resposta_ack}")
            return False

        return True

    except serial.SerialException as e:
        print(f"\n{prefixo}ERRO SERIAL (durante handshake/escrita): {e}")
        return False
    except Exception as e:
        print(f"\n{prefixo}ERRO INESPERADO (durante envio): {e}")
        return False

In [ ]:

BOOT_TIME = 3
ser = None 
try:
    prefixo = "     " 
    
    ser = serial.Serial(PORTA_SERIAL, BAUDRATE, timeout=2)
    print(f"{prefixo}Porta serial {PORTA_SERIAL} aberta com sucesso.")
    
    ser.flushInput()  
    
    print(f"{prefixo}Aguardando {BOOT_TIME}s para o boot do ESP32...")
    time.sleep(BOOT_TIME) 
    
    resposta_inicial = ser.read_all().decode('utf-8', errors='ignore')
    if resposta_inicial:
        print("\n------ [Log de Boot do ESP32] ------")
        print(resposta_inicial, end='')
        print("----------------------------------\n")
    
    print(f"{prefixo}Pronto para enviar.")


    while True:
        comando = input("\nDigite a classe (0, 1, 2) ou 's' para SAIR: ").strip().lower()

        if comando == 's':
            print(f"{prefixo}Encerrando...")
            break
        
        try:
            classe_desejada = int(comando)

            sucesso_principal = enviar_amostra_por_classe(ser, classe_desejada)
            
            if sucesso_principal:
                pass 
            else:
                print(f"{prefixo}Falha ao enviar amostra da classe {classe_desejada}.")

            if sucesso_principal:
                
                print(f"\n{prefixo}[RETORNO AUTOMÁTICO] Enviando Classe 0 (Parado)...")
                sucesso_retorno1 = enviar_amostra_por_classe(ser, 0)
                
                if sucesso_retorno1:
                    pass 
                else:
                    print(f"{prefixo}Falha ao enviar amostra da classe 0 (retorno).")

        except ValueError:
            print(f"{prefixo}Entrada inválida. Digite um número de classe ou 's'.")
        except Exception as e:
            print(f"{prefixo}Erro no loop principal: {e}")
            break 

except serial.SerialException as e:
    print(f"\n{prefixo}ERRO SERIAL (ao abrir): {e}")
    print(f"{prefixo}Verifique a porta COM, se o ESP32 está conectado e se o Monitor Serial está fechado.")
except KeyboardInterrupt:
    print(f"\n{prefixo}Envio interrompido pelo usuário.")
finally:
    if ser and ser.is_open:
        ser.close()
        print(f"\n{prefixo}Porta serial {PORTA_SERIAL} fechada.")

     Porta serial COM3 aberta com sucesso.
     Aguardando 3s para o boot do ESP32...
     Pronto para enviar.

     --- Selecionado Amostra (Índice: 2277) da Classe Real: 2 ---
     Permissão ('A') recebida. Enviando dados...
     Bytes enviados: 3200.
     Aguardando resultado da inferência (Timeout total: 10s)...

--- [RESPOSTA DO ESP32] ---
I (529020) Resultado: >>> Classe Prevista: 2 <<<
I (529020) Resultado: Classe Prevista (UART): 2
I (529020) NimBLE: GATT procedure initiated: write;
I (529020) NimBLE: att_handle=18 len=1

I (529030) BLE_CLIENTE: Enviando dados: '2'
ESP32 pronto. Aguardando comando '?'...
-----------------------------

     [RETORNO AUTOMÁTICO] Enviando Classe 0 (Parado)...

     --- Selecionado Amostra (Índice: 1576) da Classe Real: 0 ---
     Permissão ('A') recebida. Enviando dados...
     Bytes enviados: 3200.
     Aguardando resultado da inferência (Timeout total: 10s)...

--- [RESPOSTA DO ESP32] ---
I (529490) Resultado: >>> Classe Prevista: 0 <<<
I (52949